# 02 · Step-by-step exploration: NER, then targeted tone

**Scope.** German only, one block-A vote (**A-631**, Begrenzungsinitiative,
2020-09-27), every person and organisation, tone *toward* the actor only.
Each step's output is read before the next one starts.

## 1. The slice

The archive for A-631 holds every article from the 34 outlets published inside
the event window. The query filtered on outlet, dates and language, **not on
topic**. This step keeps the German rows and checks them against the pull
receipt. Nothing here prints article text, so the notebook can be committed
with its outputs.

In [3]:
import json

import pandas as pd

from mediapos import config as cfg, queries as q

EVENT_ID = "A-631"
LANGUAGE = "de"

query = pd.read_csv(cfg.DATA_OUT / "queries.csv").set_index("event_id").loc[EVENT_ID]
receipts = json.loads((cfg.DATA_OUT / "pull_receipts.json").read_text())
receipt = receipts[f"{query.date_from}_{query.date_to}"]
archive = cfg.REPO_ROOT / receipt["local_path"]

print(f"event    {EVENT_ID}  window {query.date_from} → {query.date_to}")
print(f"archive  {archive.relative_to(cfg.REPO_ROOT)}")
print(f"receipt  {receipt['articles_read']:,} articles, {receipt['by_language']}")

event    A-631  window 2020-08-28 → 2020-10-11
archive  data/corpus/2020-08-28_2020-10-11.tsv.xz
receipt  87,403 articles, {'de': 58649, 'fr': 28754}


In [4]:
articles = pd.DataFrame(r for r in q.read_articles(archive) if r["language"] == LANGUAGE)
articles["pubtime"] = pd.to_datetime(articles["pubtime"], utc=True)
articles["char_count"] = articles["char_count"].astype(int)

# The rows must be exactly what the receipt says was downloaded.
assert len(articles) == receipt["by_language"][LANGUAGE], len(articles)

print(f"{len(articles):,} {LANGUAGE} rows, count matches the receipt")
print(f"{articles['id'].nunique():,} distinct article ids")
print(f"published {articles.pubtime.min():%Y-%m-%d} → {articles.pubtime.max():%Y-%m-%d}")

58,649 de rows, count matches the receipt
58,587 distinct article ids
published 2020-08-27 → 2020-10-11


### One id, several rows: revisions of the same article

Some ids appear two or three times. These rows have the same outlet and
timestamp but a different `content_id` and body: they are **edited versions**
of one online article. Counting every version would weight an article by how
often it was edited, so we keep **one row per id: the longest version**, with
ties broken by `content_id` so the choice is deterministic.

In [5]:
versions = articles.groupby("id").size()
revised = articles[articles["id"].isin(versions[versions > 1].index)]
print(f"{(versions > 1).sum()} ids have several versions ({len(revised)} rows)")
print(revised["medium_code"].value_counts().to_string())

articles = (articles.sort_values(["id", "char_count", "content_id"], ascending=[True, False, True])
                    .drop_duplicates("id").reset_index(drop=True))
assert articles["id"].is_unique
print(f"\nslice: {len(articles):,} articles, one per id")

58 ids have several versions (120 rows)
medium_code
BLIO    106
ZWAO      6
NNTA      4
NNBU      2
SRF       2

slice: 58,587 articles, one per id


### Who published them, and in what form

In [6]:
(articles.groupby(["medium_code", "medium_name"]).size()
         .rename("articles").sort_values(ascending=False).reset_index())

,medium_code,medium_name,articles
0,AZM,Aargauer Zeitung / MLZ,4567
1,SGTO,tagblatt.ch,4392
2,BLIO,blick.ch,4239
3,NNBE,bernerzeitung.ch,4007
4,SRF,srf.ch,3601
5,NNBS,bazonline.ch,3175
6,NNTA,tagesanzeiger.ch,3149
7,ZWAO,20 minuten online,3145
8,NNBU,derbund.ch,3053
9,LUZ,Luzerner Zeitung,2881


In [7]:
articles["doctype_description"].value_counts().rename("articles").to_frame()

,articles
doctype_description,
Online medium,31540
Regional daily newspaper,14450
National daily newspaper,10767
Regional weekly newspaper,1830


In [8]:
articles["char_count"].describe(percentiles=[.1, .25, .5, .75, .9]).round(0).to_frame()

,char_count
count,58587.0
mean,4602.0
std,10304.0
min,86.0
10%,896.0
25%,1741.0
50%,3288.0
75%,5120.0
90%,7570.0
max,551056.0
